In [1]:
import torch

# Check if CUDA is available
if torch.cuda.is_available():
    # Get the number of available GPUs
    num_gpus = torch.cuda.device_count()
    print(f"Number of available GPUs: {num_gpus}")

    # Optional: print the name of each GPU
    for i in range(num_gpus):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA is not available. No GPUs found.")

Number of available GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [1]:
! pip install bitsandbytes
! pip install peft
! pip install --pre deepchem
! pip install ai2-olmo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 25.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 56.4 MB/s eta 0:00:0000:01:00:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cud

In [2]:
! pip list | grep "transformers"

sentence-transformers                    5.4.0
transformers                             5.0.0


In [4]:
%%writefile train.py
import torch
import pytorch_lightning as pl
import deepchem as dc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)
from rdkit import Chem  
# Changed to BACE Classification
from deepchem.molnet import load_bace_classification
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from pytorch_lightning.callbacks import EarlyStopping # Added EarlyStopping
# Changed metrics to Classification metrics
from sklearn.metrics import accuracy_score, roc_auc_score
from torch.nn.functional import softmax
import seaborn as sns
import re

class OlmoDataset(Dataset):
    def __init__(self, mode="Train", max_length=350): 
        self.mode = mode.lower()

        self.tokenizer = AutoTokenizer.from_pretrained(
            "harindhar10/Olmo-7b_1M_Smiles_lora",
            trust_remote_code=True,
        )
        self.tokenizer.padding_side = "right" if self.mode ==  "train" else "left"         
        self.tokenizer.pad_token = self.tokenizer.eos_token

        tasks, datasets, transformers = load_bace_classification(featurizer="raw", splitter='scaffold')
        train, valid, test = datasets

        self.task_names = tasks

        if self.mode == "train":
            self.data = train
        elif self.mode == "valid":
            self.data = valid
        elif self.mode == "test":
            self.data = test
        else:
            self.data = train

        self.max_length = max_length
        self.samples = []
        self._filldataset()

    def _filldataset(self):
        for i in range(len(self.data)):
            smiles = self.data.ids[i]
            labels = self.data.y[i]
            weights = self.data.w[i]


            for task_idx, label in enumerate(labels):
                # Filter out invalid weights
                if weights[task_idx] > 0:
                    task_name = "BACE1 inhibitor"
                    self.samples.append(self._create_prompt(smiles, task_name, label))

        print(f"[{self.mode.upper()}] Number of samples: {len(self.samples)}")

    def _create_prompt(self, smiles, task_name, label):
        eos_token = self.tokenizer.eos_token
        # Classification format: Yes (1) or No (0)
        answer = "Yes" if label == 1.0 else "No"

        full_prompt = (
            "### Instruction:\n"
            f"Is the following molecule a {task_name}?\n"
            f"<|start_of_smiles|>{smiles}<|end_of_smiles|>\n\n"
            "### Response:\n"
            f"{answer}"
        )
        return full_prompt

    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        text = self.samples[idx] 
        encodings = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        ) # here tokenizer automatically adds a eos token in the end for olmo-7b tokenizer for transformers >4.5
        input_ids = encodings["input_ids"].squeeze(0)
        attention_mask = encodings["attention_mask"].squeeze(0)
        labels = input_ids.clone()

        # Actual (non-padded) sequence length
        seq_len = attention_mask.sum().item()

        # Answer = 1 token ("true"/"false") + 1 eos = last 2 real tokens
        # Mask the entire prompt portion (everything before the answer)

        if self.tokenizer.padding_side == "right":
            labels[:seq_len - 2] = -100 # 
        else:
            labels[:-2] = -100
        # Mask padding

            
        labels[attention_mask == 0] = -100

  
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

class OLMO_QLoRA(pl.LightningModule):
    def __init__(self):
        super().__init__()

        self.tokenizer = AutoTokenizer.from_pretrained(
            "harindhar10/Olmo-7b_1M_Smiles_lora",
            trust_remote_code=True,
            padding_side="right"
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        self.peft_config = LoraConfig(
            r=32,
            lora_alpha=64,
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )
        
    def configure_model(self):
        self.model = AutoModelForCausalLM.from_pretrained(
            "harindhar10/Olmo-7b_1M_Smiles_lora",
            quantization_config=self.bnb_config,
            trust_remote_code=True,
        )
        self.model = prepare_model_for_kbit_training(self.model)
        self.model = get_peft_model(self.model, self.peft_config)
        self.model.print_trainable_parameters()

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
    def training_step(self, batch, batch_idx):
        outputs = self(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        self.log("Train_loss", loss, prog_bar=True, on_step=True, on_epoch=True, logger=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        outputs = self(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)
        return loss

    def on_train_end(self):
        if self.trainer.is_global_zero:
            print("\nStarting test set evaluation (ROC-AUC & Accuracy)...")
            
            test_dataset = OlmoDataset(mode="test", max_length=350)
            test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
            
            # Prepare special tokens for classification check
            yes_token_id = self.tokenizer.encode("Yes", add_special_tokens=False)[0]
            no_token_id = self.tokenizer.encode("No", add_special_tokens=False)[0]

            self.model.eval()
            
            y_true = []
            y_probs = [] 
            y_pred = []  

            print(f"Evaluating on {len(test_loader)} samples using Logits...")

            with torch.no_grad():
                for i, batch in enumerate(test_loader):
                    batch = {k: v.to(self.device) for k, v in batch.items()}
                    input_ids = batch["input_ids"]
                    labels = batch["labels"]
                    attention_mask = batch["attention_mask"]

                    response_mask = (labels != -100)
                    if response_mask.sum() == 0:
                        print("no answer")
                        continue
                  
                    answer_start_index = response_mask.int().argmax(dim=1).item()
                    prompt_ids = input_ids[:, :answer_start_index]
                    prompt_mask = attention_mask[:, :answer_start_index]


                    # print(self.tokenizer.decode(prompt_ids[0],skip_special_tokens=True))

                    outputs = self.model(input_ids=prompt_ids, attention_mask=prompt_mask)
                    next_token_logits = outputs.logits[:, -1, :]
                    
                    # Extract logits for "Yes" and "No"
                    yes_logit = next_token_logits[:, yes_token_id]
                    no_logit = next_token_logits[:, no_token_id]
                    
                    relevant_logits = torch.stack([no_logit, yes_logit], dim=1)
                    probs = softmax(relevant_logits, dim=1)
                    
                    # Probability of class 1 ("Yes")
                    prob_yes = probs[:, 1].item()
                    prediction = 1 if prob_yes > 0.5 else 0
                    
                    # Get True Label from the encoded labels
                    truth_id = labels[0][answer_start_index].item()
                    if truth_id == yes_token_id:
                        true_label = 1
                    elif truth_id == no_token_id:
                        true_label = 0
                    else:
                        print("no labels")
                        continue 

                    y_true.append(true_label)
                    y_probs.append(prob_yes)
                    y_pred.append(prediction)

            acc = accuracy_score(y_true, y_pred)
            try:
                roc = roc_auc_score(y_true, y_probs)
            except ValueError:
                roc = 0.0

            print("\n=== Test Set Metrics ===")
            print(f"Accuracy: {acc:.4f}")
            print(f"ROC-AUC:  {roc:.4f}")
            
            # Save the metric to the model object so it can be accessed in the main loop
            self.final_roc_auc = roc

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=1e-4, weight_decay=1e-4)

        total_steps = self.trainer.estimated_stepping_batches

        warmup_steps = int(0.15*total_steps)
        scheduler_warmup = LinearLR(
            optimizer,
            start_factor=0.01,
            end_factor=1.0,
            total_iters=warmup_steps,
        )

        scheduler_cosine = CosineAnnealingLR(
            optimizer,
            T_max=total_steps - warmup_steps,
        )

        scheduler = SequentialLR(
            optimizer,
            schedulers=[scheduler_warmup, scheduler_cosine],
            milestones=[warmup_steps]
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step"
            }
        }

if __name__ == "__main__":
    num_runs = 3
    roc_scores = []
    
    for i in range(num_runs):
        print(f"\n======================================")
        print(f"        STARTING RUN {i + 1} OF {num_runs}        ")
        print(f"======================================\n")
        
        # Change seed per run to ensure variation
        pl.seed_everything(42 , workers=True)

        train_dataset = OlmoDataset(mode="train")
        val_dataset = OlmoDataset(mode="valid")
        
        train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
        
        # Setup Early Stopping based on validation loss
        early_stop_callback = EarlyStopping(
            monitor="val_loss",
            min_delta=0.00,
            patience=3,
            verbose=True,
            mode="min"
        )
        
        trainer = pl.Trainer(
            accelerator="gpu",
            max_epochs=2,
            precision="16-mixed",
            devices = 2,
            strategy = "ddp",
            accumulate_grad_batches=8,
            enable_checkpointing=False,
            gradient_clip_val=1,
            callbacks=[early_stop_callback]
        )

        model = OLMO_QLoRA()

        # Pass both train and val loaders to fit() for EarlyStopping to work
        trainer.fit(model, train_loader, val_loader)
        
        # Extract the ROC-AUC score for this run (only on the main process)
        if trainer.is_global_zero:
            roc_scores.append(model.final_roc_auc)

    # Calculate and print Final Statistics (only on the main process)
    if torch.distributed.is_initialized():
        rank = torch.distributed.get_rank()
    else:
        rank = 0
        
    if rank == 0 and len(roc_scores) > 0:
        mean_roc = np.mean(roc_scores)
        std_roc = np.std(roc_scores)
        
        print("\n\n######################################")
        print("         FINAL TRAINING SUMMARY       ")
        print("######################################")
        print(f"Number of runs evaluated: {len(roc_scores)}")
        print(f"ROC-AUC Scores: {roc_scores}")
        print(f"Mean ROC-AUC: {mean_roc:.4f}")
        print(f"Std ROC-AUC:  {std_roc:.4f}")
        print("######################################\n")

Writing train.py


In [ ]:
! python train.py

# Classifer linear head

In [3]:
%%writefile train_linear.py
import torch
import torch.nn as nn
import pytorch_lightning as pl
import deepchem as dc
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM, # Using the base backbone
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)
from rdkit import Chem  
from deepchem.molnet import load_bace_classification
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from pytorch_lightning.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, roc_auc_score
from torch.nn.functional import softmax
from rdkit import Chem
import numpy as np

class OlmoDataset(Dataset):
    def __init__(self, mode="Train", max_length=128,do_random = False,prob = 0.2): 
        self.mode = mode.lower()

        self.tokenizer = AutoTokenizer.from_pretrained(
            "harindhar10/Olmo-7b_1M_Smiles_lora",
            trust_remote_code=True,
        )
        self.tokenizer.padding_side = "right"         
        self.tokenizer.pad_token = self.tokenizer.eos_token


        self.do_random = do_random
        self.prob = prob
        
        tasks, datasets, transformers = load_bace_classification(featurizer="raw", splitter='scaffold')
        train, valid, test = datasets

        if self.mode == "train":
            self.data = train
        elif self.mode == "valid":
            self.data = valid
        elif self.mode == "test":
            self.data = test
        else:
            self.data = train

        self.max_length = max_length
        self.samples = []
        self._filldataset()


    def _augment_smiles(self,smiles):
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None
        rand_smi = Chem.MolToSmiles(mol, doRandom=True)

        del mol # rdkit causes memory buildup 
        return rand_smi

    def _filldataset(self):
        for i in range(len(self.data)):
            smiles = self.data.ids[i]
            labels = self.data.y[i]
            weights = self.data.w[i]

            for task_idx, label in enumerate(labels):
                if weights[task_idx] > 0:
                    self.samples.append((smiles, int(label)))

        print(f"[{self.mode.upper()}] Number of samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        smiles, label = self.samples[idx]
        
        # ChemFM Style Encapsulation

        if np.random.rand() < self.prob and self.do_random:
            new_smiles = self._augment_smiles(smiles)
            smiles = new_smiles if new_smiles is not None else smiles
            
        
        text = f"<|start_of_smiles|>{smiles}<|end_of_smiles|>"
        
        encodings = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        ) 
        
        input_ids = encodings["input_ids"].squeeze(0)
        attention_mask = encodings["attention_mask"].squeeze(0)
        label_tensor = torch.tensor(label, dtype=torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": label_tensor,
        }

class OLMO_ChemFM_Classifier(pl.LightningModule):
    def __init__(self):
        super().__init__()


        self.tokenizer = AutoTokenizer.from_pretrained(
            "harindhar10/Olmo-7b_1M_Smiles_lora",
            trust_remote_code=True,
            padding_side="right"
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        self.peft_config = LoraConfig(
            r=32,
            lora_alpha=,
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
            lora_dropout=0.25,
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )
        self.loss_fn = nn.BCEWithLogitsLoss()



    def configure_model(self):
        self.model = AutoModelForCausalLM.from_pretrained(
                "harindhar10/Olmo-7b_1M_Smiles_lora",
                quantization_config=self.bnb_config,
                trust_remote_code=True,
            )
        self.model.config.pad_token_id = self.tokenizer.pad_token_id
            
        self.model = prepare_model_for_kbit_training(self.model)
        self.model = get_peft_model(self.model, self.peft_config)
        self.model.gradient_checkpointing_enable()
            
        hidden_size = self.model.config.hidden_size    
        self.classifier = nn.Sequential(
                nn.Linear(hidden_size, 1)
            )

    def forward(self, input_ids, attention_mask):
        # Forward pass through the backbone, requesting hidden states
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
            output_hidden_states=True
        )
        
        # Extract the hidden states from the final layer
        last_hidden_states = outputs.hidden_states[-1]
        
        # Find the index of the last non-padded token for each sequence in the batch
        # corresponding to <|end_of_smiles|> or the last padding token
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = input_ids.shape[0]
        
        # Extract h_l^n: the hidden state corresponding to the final token
        final_token_hidden = last_hidden_states[torch.arange(batch_size), sequence_lengths]
        
        # Pass h_l^n through the explicit linear layer
        logits = self.classifier(final_token_hidden)
        
        return logits
        
    def training_step(self, batch, batch_idx):
        logits = self(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        loss = self.loss_fn(logits.squeeze(-1), batch["labels"].float())
        self.log("Train_loss", loss, prog_bar=True, on_step=True, on_epoch=True, logger=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        logits = self(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        loss = self.loss_fn(logits.squeeze(-1), batch["labels"].float())
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)
        return loss

    def on_train_end(self):
        if not self.trainer.is_global_zero:
            return
 
        print("\n[INFO] Running test-set evaluation (ROC-AUC & Accuracy) …")
 
        test_ds     = OlmoDataset(mode="test")
        test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)
 
        self.model.eval()
        self.classifier.eval()
 
        all_probs  = []
        all_labels = []
 
        with torch.no_grad():
            for batch in test_loader:
                batch  = {k: v.to(self.device) for k, v in batch.items()}
                logits = self(batch["input_ids"], batch["attention_mask"])
                probs  = torch.sigmoid(logits).squeeze(-1).cpu().float().numpy()
                all_probs.extend(probs.tolist())
                all_labels.extend(batch["labels"].cpu().numpy().tolist())
 
        preds = [1 if p > 0.5 else 0 for p in all_probs]
        acc   = accuracy_score(all_labels, preds)
        try:
            roc = roc_auc_score(all_labels, all_probs)
        except ValueError:
            roc = 0.0
 
        print("\n=== Test Set Metrics ===")
        print(f"Accuracy : {acc:.4f}")
        print(f"ROC-AUC  : {roc:.4f}")
        self.final_roc_auc = roc

    def configure_optimizers(self):
        # Ensure the custom classifier parameters are included in the optimizer
        trainable_params = [p for p in self.parameters() if p.requires_grad]
        
        total = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Trainable params: {total}")
        optimizer = torch.optim.AdamW(trainable_params, lr=1e-5, weight_decay=1e-4)

        total_steps = self.trainer.estimated_stepping_batches
        warmup_steps = int(0.15*total_steps)
        
        scheduler_warmup = LinearLR(
            optimizer,
            start_factor=0.01,
            end_factor=1.0,
            total_iters=warmup_steps,
        )

        scheduler_cosine = CosineAnnealingLR(
            optimizer,
            T_max=total_steps - warmup_steps,
        )

        scheduler = SequentialLR(
            optimizer,
            schedulers=[scheduler_warmup, scheduler_cosine],
            milestones=[warmup_steps]
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step"
            }
        }

if __name__ == "__main__":
    num_runs = 3
    roc_scores = []
    
    for i in range(num_runs):
        print(f"\n======================================")
        print(f"        STARTING RUN {i + 1} OF {num_runs}        ")
        print(f"======================================\n")
        
        pl.seed_everything(42 + i, workers=True)

        train_dataset = OlmoDataset(mode="train",do_random = True,prob = 0.5)
        val_dataset = OlmoDataset(mode="valid")
        
        train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
        
        early_stop_callback = EarlyStopping(
            monitor="val_loss",
            min_delta=0.00,
            patience=3,
            verbose=True,
            mode="min"
        )
        
        trainer = pl.Trainer(
            accelerator="gpu",
            max_epochs=20,
            devices = 2,
            strategy = "ddp",
            precision="16-mixed",
            accumulate_grad_batches=2,
            enable_checkpointing=False,
            enable_progress_bar=False,
            gradient_clip_val=1,
            callbacks=[early_stop_callback]
        )

        model = OLMO_ChemFM_Classifier()

        trainer.fit(model, train_loader, val_loader)
        
        if trainer.is_global_zero:
            roc_scores.append(model.final_roc_auc)

    if torch.distributed.is_initialized():
        rank = torch.distributed.get_rank()
    else:
        rank = 0
        
    if rank == 0 and len(roc_scores) > 0:
        mean_roc = np.mean(roc_scores)
        std_roc = np.std(roc_scores)
        
        print("\n\n######################################")
        print("         FINAL TRAINING SUMMARY       ")
        print("######################################")
        print(f"Number of runs evaluated: {len(roc_scores)}")
        print(f"ROC-AUC Scores: {roc_scores}")
        print(f"Mean ROC-AUC: {mean_roc:.4f}")
        print(f"Std ROC-AUC:  {std_roc:.4f}")
        print("######################################\n")

Writing train_linear.py


In [ ]:
! python train_linear.py

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
2026-06-10 16:41:58.388083: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781109718.557109     143 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781109718.611014     143 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin